In [1]:
import os
from pathlib import Path

DATA_ROOT = Path("Data")
from read.springer import SpringerParser  # ✅ 确保 read/springer.py 里定义了上面的 SpringerParser


def parse_springer_article(filepath, output_folder):
    """解析单篇 Springer HTML 文件并将 paragraph 写入 txt"""
    try:
        # 1. 初始化解析器
        parser = SpringerParser(filepath)

        # 2. 解析元信息
        meta_ret = parser.parse_meta()
        if isinstance(meta_ret, dict):
            # ✅ 如果 parse_meta 返回 dict，就直接用
            meta_data = meta_ret
        else:
            # ✅ 否则，从 parser 属性里自己组装一个 dict（兼容你当前写法）
            meta_data = {
                "title": getattr(parser, "title", ""),
                "journal": getattr(parser, "journal", ""),
                "date": getattr(parser, "date", ""),
                "abstract": getattr(parser, "abstract", ""),
            }

        print(f"📄 Title: {meta_data.get('title', 'N/A')}")
        print(f"📘 Journal: {meta_data.get('journal', 'N/A')}")
        # print(f"🧾 Abstract: {meta_data.get('abstract', 'N/A')}")
        # print(f"📅 Date: {meta_data.get('date', 'N/A')}")

        # 3. 解析段落（SpringerParser.parse_paragraphs 返回的是“纯文本列表”）
        paragraph_texts = parser.parse_paragraphs()  # list[str]
        print(f"📝 段落数: {len(paragraph_texts)}")

        # 4. 构建输出文件名（去掉后缀，变成安全文件名）
        base_name = os.path.basename(filepath)
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break

        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 5. 写入到 txt 文件
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(f"Title: {meta_data.get('title', '')}\n")
            f.write(f"Journal: {meta_data.get('journal', '')}\n")
            f.write(f"Date: {meta_data.get('date', '')}\n")
            f.write(f"Abstract: {meta_data.get('abstract', '')}\n\n")
            f.write("Paragraphs:\n")
            for para in paragraph_texts:
                f.write(para + "\n\n")

        print(f"✅ 已保存段落 → {output_path}\n")

        # 如果想测试表格解析，这里可以顺便跑一遍：
        # parser.parse_tables()

    except Exception as e:
        print(f"❌ 解析失败：{filepath}\n错误信息：{e}\n")


def batch_parse_springer_folder(input_folder, output_folder):
    """批量解析整个文件夹下的 Springer HTML 文件，增加重复检查"""
    os.makedirs(output_folder, exist_ok=True)

    html_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm", ".xhtml", ".xml"))
    ]

    if not html_files:
        print("⚠️ 未找到 HTML/XML 文件，请检查路径。")
        return

    print(f"🚀 开始批量解析 Springer 文献，共 {len(html_files)} 篇...\n")

    for html_file in html_files:
        # --- 核心改进：预先计算输出文件名并检查 ---
        base_name = html_file
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break
        
        # 生成与 parse_springer_article 内部完全一致的安全文件名
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 检查是否已存在
        if os.path.exists(output_path):
            print(f"⏭️  跳过已存在: {safe_name}.txt")
            continue
        # ----------------------------------------

        file_path = os.path.join(input_folder, html_file)
        parse_springer_article(file_path, output_folder)

    print("🎯 全部解析完成！")


if __name__ == "__main__":
    # 👉 换成你的 Springer HTML 文件夹路径
    input_folder = DATA_ROOT / "Springer" / "source"   # 📂 输入 HTML/XML 文件夹路径
    output_folder = DATA_ROOT / "Springer" / "txt"     # 📂 输出 TXT 文件夹路径

    batch_parse_springer_folder(input_folder, output_folder)


🚀 开始批量解析 Springer 文献，共 300 篇...

📄 Title: The effect of chain extenders structure on properties of new polyurethane elastomers
📘 Journal: 
✅ Found 280 paragraphs.

[1] Access provided by Donghua University...

[2] 2021 Accesses...

[3] 51 Citations...

[4] 3 
                                Altmetric...

[5] Explore all metrics...

[6] Two series of polyurethane elastomers were synthesized to investigate what effect does the incorporation of various new chain extenders have on the me...

[7] Avoid common mistakes on your manuscript....

[8] Polyurethanes (PUs) are an important class of materials with wide application such as in coatings, binder resins, fibers, and high-performance elastom...

[9] For potential utility in biomedical applications it is required to obtain good biocompatibility and low hydrolytic and enzymatic degradable tendency. ...

[10] It is well-known that inter-urethane hydrogen bonding between the carbonyl and N–H groups and microphase composition are important par